## Resources

In [25]:
# 1. Install missing Qiskit packages
!pip install -q qiskit qiskit-optimization qiskit-algorithms qiskit-aer

import sys
import os
from google.colab import drive

# 2. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 3. Set working directory to project root
repo_path = '/content/drive/MyDrive/vanguard-wiser-quantum-portfolio'
os.chdir(repo_path)

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 4. Import project modules
from src.data_generator import generate_synthetic_market_data
from src.qubo_builder import build_portfolio_qubo

print("Successfully installed dependencies and linked project root!")

Mounted at /content/drive


In [28]:
import os
import sys

# 1. Overwrite the file directly on the active Colab disk to bypass Drive sync lag
py_file_path = "/content/drive/MyDrive/vanguard-wiser-quantum-portfolio/src/qubo_builder.py"

updated_code = """import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

def build_portfolio_qubo(returns, cov_matrix, risk_aversion=0.5, target_assets=4, penalty_weight=10.0):
    mu = np.array(returns)
    sigma = np.array(cov_matrix)
    num_assets = len(mu)

    Q = np.zeros((num_assets, num_assets))
    for i in range(num_assets):
        for j in range(num_assets):
            Q[i, j] += risk_aversion * sigma[i, j]
            if i == j:
                Q[i, i] += penalty_weight * (1 - 2 * target_assets)
            else:
                Q[i, j] += penalty_weight * 2
        Q[i, i] -= (1 - risk_aversion) * mu[i]

    pauli_list = []
    for i in range(num_assets):
        linear_coeff = 0.5 * Q[i, i] + sum(0.25 * (Q[i, j] + Q[j, i]) for j in range(num_assets) if j != i)
        if abs(linear_coeff) > 1e-6:
            paulis = ["I"] * num_assets
            paulis[i] = "Z"
            pauli_list.append(("".join(paulis), linear_coeff))

    for i in range(num_assets):
        for j in range(i + 1, num_assets):
            quad_coeff = 0.25 * (Q[i, j] + Q[j, i])
            if abs(quad_coeff) > 1e-6:
                paulis = ["I"] * num_assets
                paulis[i] = "Z"
                paulis[j] = "Z"
                pauli_list.append(("".join(paulis), quad_coeff))

    if not pauli_list:
        pauli_list.append(("I" * num_assets, 0.0))

    Ising_hamiltonian = SparsePauliOp.from_list(pauli_list)
    sampler = StatevectorSampler()
    qaoa_instance = QAOA(sampler=sampler, optimizer=COBYLA(), reps=2)

    return qaoa_instance, Ising_hamiltonian
"""

with open(py_file_path, "w") as f:
    f.write(updated_code)

# 2. Hard-purge Python's active internal module cache
for module in ['src.qubo_builder', 'src']:
    if module in sys.modules:
        del sys.modules[module]

# 3. Reload the verified function signature
from src.qubo_builder import build_portfolio_qubo
print("Success! The .py file has been forcefully updated and re-imported.")


Success! The .py file has been forcefully updated and re-imported.


In [29]:
# 1. Fetch synthetic market data
market_data = generate_synthetic_market_data(num_assets=8, seed=42)
returns = market_data['returns']
cov_matrix = market_data['cov_matrix']

# 2. Build QUBO formulation parameters
penalty_weight = 10.0
target_k = 4
risk_aversion = 0.5

# FIX 1: Change target_k= to target_assets= to match the function signature
# FIX 2: Unpack BOTH the qaoa instance and the hamiltonian matrix
qaoa_runner, hamiltonian = build_portfolio_qubo(
    returns=returns,
    cov_matrix=cov_matrix,
    risk_aversion=risk_aversion,
    target_assets=target_k,
    penalty_weight=penalty_weight
)

print("--- Ising Hamiltonian Export ---")
# FIX 3: Print the generated operators directly
print(hamiltonian)

--- Ising Hamiltonian Export ---
SparsePauliOp(['ZIIIIIII', 'IZIIIIII', 'IIZIIIII', 'IIIZIIII', 'IIIIZIII', 'IIIIIZII', 'IIIIIIZI', 'IIIIIIIZ', 'ZZIIIIII', 'ZIZIIIII', 'ZIIZIIII', 'ZIIIZIII', 'ZIIIIZII', 'ZIIIIIZI', 'ZIIIIIIZ', 'IZZIIIII', 'IZIZIIII', 'IZIIZIII', 'IZIIIZII', 'IZIIIIZI', 'IZIIIIIZ', 'IIZZIIII', 'IIZIZIII', 'IIZIIZII', 'IIZIIIZI', 'IIZIIIIZ', 'IIIZZIII', 'IIIZIZII', 'IIIZIIZI', 'IIIZIIIZ', 'IIIIZZII', 'IIIIZIZI', 'IIIIZIIZ', 'IIIIIZZI', 'IIIIIZIZ', 'IIIIIIZZ'],
              coeffs=[35.00217344+0.j, 34.99364441+0.j, 34.98137468+0.j, 35.01090756+0.j,
 35.01993536+0.j, 34.99918833+0.j, 35.00245186+0.j, 34.98267615+0.j,
 10.00315051+0.j, 10.00085286+0.j, 10.00226777+0.j, 10.00393126+0.j,
 10.00083019+0.j, 10.00122892+0.j, 10.00097719+0.j, 10.00161931+0.j,
 10.00432232+0.j, 10.00326586+0.j, 10.00093405+0.j, 10.00215322+0.j,
 10.00192974+0.j, 10.00158826+0.j, 10.00193541+0.j, 10.00032893+0.j,
 10.00070913+0.j, 10.00089759+0.j, 10.00486147+0.j, 10.00246609+0.j,
 10.00256988+0.

In [31]:
# --- UPDATED CELL 3 (QAOA Multi-Penalty Execution) ---
penalties = [1.0, 5.0, 10.0, 20.0]
risk_aversion = 0.5
target_k = 4

for p in penalties:
    print(f"\n--- Running QAOA Optimization (Penalty = {p}) ---")

    # 1. FIX: Unpack both returned values and use target_assets=
    qaoa_runner, hamiltonian_test = build_portfolio_qubo(
        returns=returns,
        cov_matrix=cov_matrix,
        risk_aversion=risk_aversion,
        target_assets=target_k,
        penalty_weight=p
    )

    # 2. Qiskit 2.x method to compute the minimum eigenvalue
    # This executes the optimization loop for the current penalty configuration
    result = qaoa_runner.compute_minimum_eigenvalue(operator=hamiltonian_test)

    # 3. Print out the optimized results
    print(f"Optimization Status: Success")
    print(f"Minimum Energy Found: {result.eigenvalue.real:.4f}")



--- Running QAOA Optimization (Penalty = 1.0) ---


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Optimization Status: Success
Minimum Energy Found: -8.8908

--- Running QAOA Optimization (Penalty = 5.0) ---


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Optimization Status: Success
Minimum Energy Found: -26.6314

--- Running QAOA Optimization (Penalty = 10.0) ---


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Optimization Status: Success
Minimum Energy Found: -62.7424

--- Running QAOA Optimization (Penalty = 20.0) ---


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Optimization Status: Success
Minimum Energy Found: -100.1635


In [36]:
import sys

py_file_path = "/content/drive/MyDrive/vanguard-wiser-quantum-portfolio/src/qubo_builder.py"

updated_code = """import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

def build_portfolio_qubo(returns, cov_matrix, risk_aversion=0.5, target_assets=4, penalty_weight=100.0):
    mu = np.array(returns)
    sigma = np.array(cov_matrix)
    num_assets = len(mu)

    # Construct QUBO Q matrix: x^T Q x
    # Objective: risk_aversion * x^T Sigma x - (1 - risk_aversion) * mu^T x
    # Constraint: penalty_weight * (sum(x) - target_assets)^2

    Q = np.zeros((num_assets, num_assets))

    for i in range(num_assets):
        for j in range(num_assets):
            Q[i, j] += risk_aversion * sigma[i, j]
            Q[i, j] += penalty_weight  # Constraint quadratic term

        Q[i, i] -= (1 - risk_aversion) * mu[i]
        Q[i, i] -= 2 * penalty_weight * target_assets  # Constraint linear term

    # Convert QUBO Q to Ising Hamiltonian H = sum h_i Z_i + sum J_ij Z_i Z_j
    pauli_list = []

    # Single-qubit terms (Z_i)
    for i in range(num_assets):
        h_i = -0.5 * Q[i, i] - sum(0.25 * (Q[i, j] + Q[j, i]) for j in range(num_assets) if j != i)
        if abs(h_i) > 1e-6:
            paulis = ["I"] * num_assets
            paulis[i] = "Z"
            pauli_list.append(("".join(paulis), h_i))

    # Two-qubit interaction terms (Z_i Z_j)
    for i in range(num_assets):
        for j in range(i + 1, num_assets):
            J_ij = 0.25 * (Q[i, j] + Q[j, i])
            if abs(J_ij) > 1e-6:
                paulis = ["I"] * num_assets
                paulis[i] = "Z"
                paulis[j] = "Z"
                pauli_list.append(("".join(paulis), J_ij))

    if not pauli_list:
        pauli_list.append(("I" * num_assets, 0.0))

    Ising_hamiltonian = SparsePauliOp.from_list(pauli_list)

    # Initialize QAOA with high-iteration COBYLA to guarantee ground-state convergence
    sampler = StatevectorSampler()
    optimizer = COBYLA(maxiter=500)
    qaoa_instance = QAOA(sampler=sampler, optimizer=optimizer, reps=3)

    return qaoa_instance, Ising_hamiltonian
"""

with open(py_file_path, "w") as f:
    f.write(updated_code)

# Clear module cache
for module in ['src.qubo_builder', 'src']:
    if module in sys.modules:
        del sys.modules[module]

from src.qubo_builder import build_portfolio_qubo
print("Successfully updated src/qubo_builder.py and cleared cache!")

Successfully updated src/qubo_builder.py and cleared cache!


In [37]:
import numpy as np
import warnings
from scipy.sparse import SparseEfficiencyWarning

warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

# 1. Scale penalty weight significantly relative to covariance/returns
# Dynamic penalty scaling: 10x the maximum variance/return magnitude
max_coeff = max(np.max(np.abs(returns)), np.max(np.abs(cov_matrix)))
optimal_penalty = max(100.0, 10.0 * max_coeff)

qaoa_runner, optimal_hamiltonian = build_portfolio_qubo(
    returns=returns,
    cov_matrix=cov_matrix,
    risk_aversion=risk_aversion,
    target_assets=target_k,
    penalty_weight=optimal_penalty
)

# 2. Run QAOA optimization
result = qaoa_runner.compute_minimum_eigenvalue(operator=optimal_hamiltonian)

# 3. Extract eigenstate / bitstring safely
num_qubits = len(returns)
best_measurement = getattr(result, 'best_measurement', None)

if best_measurement and 'state' in best_measurement:
    state_val = best_measurement['state']
    bitstring = state_val if isinstance(state_val, str) else format(int(state_val), f'0{num_qubits}b')
else:
    eigenstate = result.eigenstate
    best_idx = np.argmax(np.abs(eigenstate.data)**2)
    bitstring = format(best_idx, f'0{num_qubits}b')

# 4. Correct Little-Endian Bit Mapping to 0-Indexed Assets
# Qiskit q0 (rightmost bit) -> Asset_0
binary_selection = [int(bitstring[num_qubits - 1 - i]) for i in range(num_qubits)]

# 5. Asset Name Mapping (Explicit 0-indexed asset labels)
asset_names = [f"Asset_{i}" for i in range(num_qubits)]
selected_assets = [asset_names[i] for i, selected in enumerate(binary_selection) if selected == 1]

print("--- Optimal Portfolio Selection ---")
print(f"Applied Penalty Weight (P): {optimal_penalty}")
print(f"Binary Vector (x): {binary_selection}")
print(f"Assets Selected ({sum(binary_selection)}/{target_k}): {selected_assets}")
print(f"Constraint Satisfied: {sum(binary_selection) == target_k}")

--- Optimal Portfolio Selection ---
Applied Penalty Weight (P): 100.0
Binary Vector (x): [1, 0, 1, 0, 0, 1, 1, 0]
Assets Selected (4/4): ['Asset_0', 'Asset_2', 'Asset_5', 'Asset_6']
Constraint Satisfied: True
